# Process Window Scout (AM-PWS)

An agent that recommends **PBF-LB process windows** from the open-access literature and tells you which
published results to trust. Example request: *"316L, 200 W machine, 30 µm layers, density above 99.5 %"*.

This notebook is the whole project. Each part adds one piece of the agent, explains why it is built that
way, and tests it in the cells right below. Parts follow the course modules, so a concept is learned before
it is used.

**How to run:** kernel *Python 3.13 (CMU Agents)*, then *Kernel > Restart & Run All*. Every test cell prints
`ok` lines and stops with an error if something breaks. Slow steps (paper downloads, model calls) will save
their results in `data/` so a rerun is quick.

| Part | What it adds | Course module | Status |
|---|---|---|---|
| 1 | Deterministic tools: units, energy density, melt-pool band | 2 (tools) | built |
| 2 | Local model (Ollama) and a first ReAct agent with a trace log | 2 (ReAct) | next |
| 3 | Paper pipeline: OpenAlex search, PDF parsing, evidence store | 3 (RAG) | planned |
| 4 | Source check, consensus and conflict check | 3 and 4 | planned |
| 5 | Tree of Thought for outliers and window selection | 4 | planned |
| 6 | Reader and Checker agents, tools over MCP | 5 | planned |
| 7 | Guardrails, clarification step, engineer feedback memory | 6 | planned |
| 8 | Evaluation: prompt-only baseline vs agent | 7 | planned |

## 0. Setup

Paths and one small helper. `check()` is my test function: it prints `ok` when a condition holds and raises
an error when it does not, so *Restart & Run All* stops at the first broken piece.

In [ ]:
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import lambertw

# Project root = the repository folder (this notebook lives in notebooks/).
ROOT = Path.cwd() if (Path.cwd() / "docs").exists() else Path.cwd().parent
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
print("Project root:", ROOT)


def check(name, condition):
    # Tiny test helper: print ok, or stop the notebook with a clear message.
    assert condition, f"FAILED: {name}"
    print(f"  ok  {name}")

## Part 1. Deterministic tools

**Why tools first.** The agent will read numbers from papers and compare them. Language models are poor at
arithmetic (Module 1: numbers are split into tokens, and the model pattern-matches rather than calculates), so
the rule for this project is: *the model never does arithmetic*. Every number it reasons about comes from a
paper or from one of these plain Python functions.

Each tool follows the same contract (from Lab 2.2):
- typed inputs with the unit in the argument name (`power_W`, `scan_speed_mm_s`);
- a docstring that says what the tool does, when to use it, its inputs with units and an example. In
  Part 2 the agent framework reads the name, type hints and docstring to describe the tool to the model,
  so this text decides whether the model picks the right tool;
- it returns a dictionary: numbers for my code, plus a `summary` sentence that restates the inputs, units and
  result, which is what the agent will read;
- invalid input raises `ValueError` with a readable message. In Part 2 the agent loop will turn that message
  into an observation, so the model can correct itself instead of crashing;
- values that are legal but unusual for PBF-LB produce a warning in the summary. A scan speed of 0.8 "mm/s" is
  almost certainly 0.8 m/s, and I want that caught before it spoils a comparison.

The tools are tested here without any model. Only once they pass does an agent get to use them.

### 1.1 Unit conversion

Papers report scan speed in mm/s, m/s or even mm/min, and layer thickness in µm or mm. Before two papers can be
compared, everything goes to one set of units: **W, mm/s, µm**. The table maps each unit to its dimension and a
factor to SI, so a conversion is `value * factor_from / factor_to`. Asking for W to mm is refused rather than
silently producing nonsense.

In [ ]:
# unit -> (dimension, factor to SI)
UNITS = {
    "W": ("power", 1.0), "kW": ("power", 1e3),
    "mm/s": ("speed", 1e-3), "m/s": ("speed", 1.0),
    "mm/min": ("speed", 1e-3 / 60), "m/min": ("speed", 1 / 60),
    "um": ("length", 1e-6), "µm": ("length", 1e-6), "μm": ("length", 1e-6),  # micro sign and Greek mu
    "mm": ("length", 1e-3), "m": ("length", 1.0),
    "J/mm3": ("energy density", 1e9), "J/m3": ("energy density", 1.0),
}


def convert_units(value: float, from_unit: str, to_unit: str) -> dict:
    """Convert a power, speed, length or energy density value from one unit to another.

    Use this whenever a paper reports a value in a different unit from the one needed for a comparison
    (project units are W, mm/s and µm). Do not convert units yourself.

    Args:
        value: the number to convert.
        from_unit: unit of the value, one of W, kW, mm/s, m/s, mm/min, m/min, um, µm, mm, m, J/mm3, J/m3.
        to_unit: unit wanted, same list; must measure the same quantity as from_unit.

    Example: convert_units(0.8, "m/s", "mm/s") gives 800 mm/s.
    Returns a dict with "value", "unit" and a "summary" sentence.
    """
    for u in (from_unit, to_unit):
        if u not in UNITS:
            raise ValueError(f"Unknown unit '{u}'. Known units: {', '.join(UNITS)}.")
    dim_from, f_from = UNITS[from_unit]
    dim_to, f_to = UNITS[to_unit]
    if dim_from != dim_to:
        raise ValueError(f"Cannot convert {dim_from} ({from_unit}) to {dim_to} ({to_unit}).")
    result = value * f_from / f_to
    return {
        "value": result,
        "unit": to_unit,
        "summary": f"{value:g} {from_unit} = {result:.6g} {to_unit} ({dim_from}).",
    }


print(convert_units(0.8, "m/s", "mm/s")["summary"])
print(convert_units(0.03, "mm", "µm")["summary"])

In [ ]:
# Tests for convert_units
check("0.8 m/s is 800 mm/s", math.isclose(convert_units(0.8, "m/s", "mm/s")["value"], 800))
check("48 m/min is 800 mm/s", math.isclose(convert_units(48, "m/min", "mm/s")["value"], 800))
check("0.03 mm is 30 µm", math.isclose(convert_units(0.03, "mm", "µm")["value"], 30))
check("micro sign and Greek mu agree", convert_units(1, "mm", "µm")["value"] == convert_units(1, "mm", "μm")["value"])
check("0.4 kW is 400 W", math.isclose(convert_units(0.4, "kW", "W")["value"], 400))

for bad in [(1, "W", "mm"), (1, "furlong", "m")]:
    try:
        convert_units(*bad)
        check(f"{bad} should raise", False)
    except ValueError as e:
        check(f"{bad} raises: {e}", True)

### 1.2 Energy density

Volumetric energy density is the number most papers use to summarise a parameter set:

$$E_v = \frac{P}{v \, h \, t} \quad [\mathrm{J/mm^3}]$$

with laser power $P$ (W), scan speed $v$ (mm/s), hatch spacing $h$ (mm) and layer thickness $t$ (mm). The tool
also reports linear energy density $P/v$ (J/mm).

$E_v$ is a coarse metric: two parameter sets with the same $E_v$ can give very different melt pools and
densities (Bertoli et al., 2017). The agent may use it to group papers, never to judge a result on its own,
and the summary says so.

The helper `_range_warnings` holds what I consider typical PBF-LB ranges. Values outside them are allowed but
flagged, mainly to catch unit mistakes.

In [ ]:
# Typical PBF-LB ranges: outside these is legal but suspicious (often a unit mistake).
TYPICAL = {
    "power_W": (20, 1000),
    "scan_speed_mm_s": (50, 5000),
    "hatch_um": (20, 300),
    "layer_um": (10, 150),
}


def _validate(**inputs) -> list[str]:
    # Raise on non-positive values; return warnings for values outside typical PBF-LB ranges.
    warnings = []
    for name, value in inputs.items():
        if value is None or not value > 0:
            raise ValueError(f"{name} must be a positive number, got {value!r}.")
        lo, hi = TYPICAL[name]
        if not lo <= value <= hi:
            warnings.append(f"{name} = {value:g} is outside the typical PBF-LB range {lo:g} to {hi:g}; check the units.")
    return warnings


def energy_density(power_W: float, scan_speed_mm_s: float, hatch_um: float, layer_um: float) -> dict:
    """Volumetric energy density E_v = P/(v*h*t) in J/mm³ and linear energy density P/v in J/mm.

    Use this when a paper gives laser power, scan speed, hatch spacing and layer thickness and you need
    its energy density, for example to group papers. E_v is a coarse metric: never judge a result on
    E_v alone.

    Args:
        power_W: laser power in W.
        scan_speed_mm_s: scan speed in mm/s (convert m/s first).
        hatch_um: hatch spacing in µm.
        layer_um: layer thickness in µm.

    Example: energy_density(200, 800, 100, 30) gives E_v = 83.3 J/mm³.
    Returns a dict with "ved_J_mm3", "led_J_mm", "warnings" and a "summary" sentence.
    """
    warnings = _validate(power_W=power_W, scan_speed_mm_s=scan_speed_mm_s, hatch_um=hatch_um, layer_um=layer_um)
    ved = power_W / (scan_speed_mm_s * (hatch_um / 1000) * (layer_um / 1000))
    led = power_W / scan_speed_mm_s
    summary = (
        f"For P = {power_W:g} W, v = {scan_speed_mm_s:g} mm/s, h = {hatch_um:g} µm, t = {layer_um:g} µm: "
        f"volumetric energy density E_v = P/(v*h*t) = {ved:.1f} J/mm³; linear energy density P/v = {led:.3f} J/mm. "
        "E_v is a coarse metric: equal E_v does not mean an equal melt pool or density."
    )
    if warnings:
        summary += " Warning: " + " ".join(warnings)
    return {"ved_J_mm3": ved, "led_J_mm": led, "warnings": warnings, "summary": summary}


print(energy_density(200, 800, 100, 30)["summary"])

In [ ]:
# Tests for energy_density
r = energy_density(200, 800, 100, 30)
check("200 W, 800 mm/s, 100 µm, 30 µm gives 83.3 J/mm³", math.isclose(r["ved_J_mm3"], 83.333, rel_tol=1e-4))
check("linear energy density 0.25 J/mm", math.isclose(r["led_J_mm"], 0.25))
check("no warnings for a typical set", r["warnings"] == [])
check("doubling speed halves E_v", math.isclose(energy_density(200, 1600, 100, 30)["ved_J_mm3"], r["ved_J_mm3"] / 2))

# A speed given in m/s by mistake: allowed, but flagged in the summary the agent reads.
r_bad = energy_density(200, 0.8, 100, 30)
check("0.8 mm/s is flagged as a likely unit mistake", "scan_speed_mm_s" in r_bad["summary"] and r_bad["warnings"])

for bad in [dict(power_W=0, scan_speed_mm_s=800, hatch_um=100, layer_um=30),
            dict(power_W=200, scan_speed_mm_s=-5, hatch_um=100, layer_um=30)]:
    try:
        energy_density(**bad)
        check(f"{bad} should raise", False)
    except ValueError as e:
        check(f"raises: {e}", True)

### 1.3 Melt-pool band (physics as weak evidence)

This is the tool that lets the agent *generate new data* rather than only read it. It estimates melt-pool width
and depth with the **Rosenthal** solution for a moving point heat source on a semi-infinite body. In the frame
moving with the laser (position $\xi$ along the scan direction, $\xi < 0$ behind the laser, $R$ the distance
from the source):

$$T - T_0 = \frac{A P}{2 \pi k R} \exp\left(-\frac{v (\xi + R)}{2 \alpha}\right), \qquad \alpha = \frac{k}{\rho c_p}$$

The melt pool is the region where $T \ge T_m$. For a given $\xi$ the right-hand side falls steadily with $R$,
so the melt boundary has a closed form through the Lambert W function. With $b = v / 2\alpha$ and
$C = A P / (2 \pi k (T_m - T_0))$:

$$R_m(\xi) = \frac{1}{b} W_0\left(b \, C \, e^{-b \xi}\right), \qquad r(\xi) = \sqrt{R_m^2 - \xi^2}$$

The pool runs from the rear tip at $\xi = -C$ to the front tip at $\xi_f = W_0(2 b C)/(2b)$. Width is
$2 \max r$. Because only $R$ appears in the solution, the cross-section is a half circle: **depth = width / 2**.

**Why a band, and why weak evidence.** Rosenthal ignores latent heat, Marangoni flow, keyholing, the powder
layer and temperature-dependent properties, and the absorptivity $A$ is rarely known. I run it at
$A$ = 0.3, 0.5 and 0.7 and treat the spread as an uncertainty band. In Part 4 a paper is flagged as an outlier
only if the consensus of other papers *and the whole band* disagree with it. In keyhole conditions the real
depth will be well above this estimate, so a too-shallow band is expected there and is not a contradiction.

**Material properties** are averages between room temperature and melting. They are approximate (after Mills,
2002) and are the first thing to revisit if the band looks off.

In [ ]:
# Approximate 316L properties, averaged between room temperature and melting (after Mills, 2002).
MATERIALS = {
    "316L": {
        "k_W_mK": 22.0,      # thermal conductivity (about 14 at room temperature, about 30 near melting)
        "rho_kg_m3": 7800.0, # density
        "cp_J_kgK": 600.0,   # specific heat (about 500 to 720 over the range)
        "T_melt_K": 1700.0,  # between solidus (about 1658 K) and liquidus (about 1723 K)
        "T0_K": 300.0,       # build plate / powder bed temperature, no preheat
    },
}


def _rosenthal_pool(power_W, scan_speed_mm_s, absorptivity, mat):
    # Melt-pool width, depth and length (µm) from the Rosenthal point-source solution.
    k, rho, cp = mat["k_W_mK"], mat["rho_kg_m3"], mat["cp_J_kgK"]
    v = scan_speed_mm_s / 1000                      # m/s
    alpha = k / (rho * cp)                          # thermal diffusivity, m²/s
    b = v / (2 * alpha)                             # 1/m
    C = absorptivity * power_W / (2 * math.pi * k * (mat["T_melt_K"] - mat["T0_K"]))  # m

    xi_rear = -C                                    # rear tip of the pool
    xi_front = lambertw(2 * b * C).real / (2 * b)   # front tip of the pool
    xi = np.linspace(xi_rear, xi_front, 4001)
    R_m = lambertw(b * C * np.exp(-b * xi)).real / b   # melt boundary distance at each xi
    r = np.sqrt(np.clip(R_m**2 - xi**2, 0, None))      # half-width of the pool at each xi

    width = 2 * r.max()
    return {"width_um": width * 1e6, "depth_um": width / 2 * 1e6, "length_um": (xi_front - xi_rear) * 1e6}


def melt_pool_band(power_W: float, scan_speed_mm_s: float, material: str = "316L",
                   absorptivities: tuple = (0.3, 0.5, 0.7)) -> dict:
    """Estimated melt-pool width and depth (µm) from the Rosenthal model, as a band over absorptivity.

    Use this to see whether physics supports or contradicts a reported result at a given laser power and
    scan speed, for example whether the melt pool is likely deeper than the layer. The result is weak
    evidence: the model ignores fluid flow and keyholing, so treat the band as a rough range, not a check.

    Args:
        power_W: laser power in W.
        scan_speed_mm_s: scan speed in mm/s (convert m/s first).
        material: alloy name; only "316L" is available.
        absorptivities: laser absorptivities to try, each between 0 and 1 (default 0.3, 0.5, 0.7).

    Example: melt_pool_band(200, 800) gives width about 101 to 156 µm, depth about 51 to 78 µm.
    Returns a dict with "material", "band" (one row per absorptivity), "warnings" and a "summary" sentence.
    """
    if material not in MATERIALS:
        raise ValueError(f"No properties for material '{material}'. Available: {', '.join(MATERIALS)}.")
    for a in absorptivities:
        if not 0 < a <= 1:
            raise ValueError(f"Absorptivity must be between 0 and 1, got {a}.")
    warnings = _validate(power_W=power_W, scan_speed_mm_s=scan_speed_mm_s)

    rows = [{"absorptivity": a, **_rosenthal_pool(power_W, scan_speed_mm_s, a, MATERIALS[material])}
            for a in absorptivities]
    w = [row["width_um"] for row in rows]
    d = [row["depth_um"] for row in rows]
    summary = (
        f"Rosenthal estimate for {material} at P = {power_W:g} W, v = {scan_speed_mm_s:g} mm/s, "
        f"absorptivity {min(absorptivities):g} to {max(absorptivities):g}: melt-pool width {min(w):.0f} to {max(w):.0f} µm, "
        f"depth {min(d):.0f} to {max(d):.0f} µm. Weak evidence: the model ignores fluid flow and keyholing, "
        "so real depth can be much larger at high energy input."
    )
    if warnings:
        summary += " Warning: " + " ".join(warnings)
    return {"material": material, "band": rows, "warnings": warnings, "summary": summary}


band = melt_pool_band(200, 800)
print(band["summary"])
for row in band["band"]:
    print(f"  A = {row['absorptivity']}: width {row['width_um']:.0f} µm, depth {row['depth_um']:.0f} µm, length {row['length_um']:.0f} µm")

**How do I know the solver is right?** Three kinds of test:
1. *Physical trends:* the pool grows with power and absorptivity and shrinks with speed.
2. *Built-in geometry:* depth is exactly half the width.
3. *An independent formula:* at high speed the Rosenthal width tends to
   $W \approx \sqrt{\dfrac{8 A P}{\pi e \, \rho c_p v (T_m - T_0)}}$, which I derived separately from the same
   equation. My numerical result should approach it as speed increases, and stay within a few percent at
   typical PBF-LB speeds.

In [ ]:
# Tests for melt_pool_band
def width(P, v, A=0.5):
    return _rosenthal_pool(P, v, A, MATERIALS["316L"])["width_um"]

def width_high_speed(P, v, A=0.5, m=MATERIALS["316L"]):
    dT = m["T_melt_K"] - m["T0_K"]
    return math.sqrt(8 * A * P / (math.pi * math.e * m["rho_kg_m3"] * m["cp_J_kgK"] * (v / 1000) * dT)) * 1e6

check("wider at higher power", width(300, 800) > width(200, 800))
check("narrower at higher speed", width(200, 1200) < width(200, 800))
check("wider at higher absorptivity", width(200, 800, 0.7) > width(200, 800, 0.3))

pool = _rosenthal_pool(200, 800, 0.5, MATERIALS["316L"])
check("depth is half the width", math.isclose(pool["depth_um"], pool["width_um"] / 2))

for P, v in [(200, 800), (370, 1500), (200, 3000)]:
    ratio = width(P, v) / width_high_speed(P, v)
    check(f"P = {P} W, v = {v} mm/s: within 5 % of the high-speed formula (ratio {ratio:.3f})", abs(ratio - 1) < 0.05)
check("closer to the formula at higher speed", abs(width(200, 3000) / width_high_speed(200, 3000) - 1)
      < abs(width(200, 400) / width_high_speed(200, 400) - 1))

b = melt_pool_band(200, 800)
check("band has three absorptivities", [row["absorptivity"] for row in b["band"]] == [0.3, 0.5, 0.7])
check("summary states inputs and units", "200 W" in b["summary"] and "800 mm/s" in b["summary"] and "µm" in b["summary"])
for bad in [dict(material="Ti64"), dict(absorptivities=(0.5, 1.5))]:
    try:
        melt_pool_band(200, 800, **bad)
        check(f"{bad} should raise", False)
    except ValueError as e:
        check(f"raises: {e}", True)

**Demo: the band across scan speeds.** The shaded area is what the agent will later hold a paper's reported
result against. At 30 µm layers, for example, a depth band that sits well above the layer thickness says the
physics does not argue against full fusion; it does not prove it.

In [ ]:
speeds = np.linspace(300, 2000, 30)
fig, ax = plt.subplots(figsize=(7, 4))
for P, colour in [(150, "tab:blue"), (200, "tab:orange"), (300, "tab:green")]:
    bands = [melt_pool_band(P, v)["band"] for v in speeds]
    lo = [b_[0]["depth_um"] for b_ in bands]
    mid = [b_[1]["depth_um"] for b_ in bands]
    hi = [b_[2]["depth_um"] for b_ in bands]
    ax.fill_between(speeds, lo, hi, color=colour, alpha=0.25)
    ax.plot(speeds, mid, color=colour, label=f"{P} W (A = 0.5 line, band 0.3 to 0.7)")
ax.axhline(30, color="grey", ls="--", lw=1)
ax.text(1950, 32, "30 µm layer", ha="right", color="grey")
ax.set_ylim(0, None)
ax.set_xlabel("Scan speed (mm/s)")
ax.set_ylabel("Rosenthal melt-pool depth (µm)")
ax.set_title("316L: Rosenthal depth band (weak evidence)")
ax.legend()
plt.show()

### Part 1 summary

- Three tested tools: `convert_units`, `energy_density`, `melt_pool_band`. None needs a model.
- Each returns numbers for code and a sentence with inputs and units for the agent.
- Bad inputs raise readable errors; suspicious ones are flagged in the sentence.
- The melt-pool solver agrees with an independent high-speed formula, but its output stays weak evidence.

**Next (Part 2):** install Ollama, pick a first local model, and write a ReAct loop that answers an
energy-density question by calling these tools, with every thought, tool call and observation logged.